<a name="top"></a><img src="../images/chisel_1024.png" alt="Chisel logo" style="width:480px;" />

# 模块 3.5：面向对象编程
**上一节：[函数式编程](3.4_functional_programming.ipynb)**<br>
**下一节：[类型](3.6_types.ipynb)**

## 动机
Scala 和 Chisel 是面向对象的编程语言，这意味着代码可以被封装到对象中。
Scala 构建在 Java 之上，继承了 Java 的许多面向对象特性。
然而，正如我们将在下面看到的，也存在一些差异。
Chisel 的硬件模块与 Verilog 的模块类似，它们可以被实例化，并以单个或多个实例的方式连接起来。

## 环境设置

In [ ]:
val path = System.getProperty("user.dir") + "/../source/load-ivy.sc"
interp.load.module(ammonite.ops.Path(java.nio.file.FileSystems.getDefault().getPath(path)))

In [ ]:
import chisel3._
import chisel3.util._
import chisel3.experimental._
import chisel3.tester._
import chisel3.tester.RawTester.test

---
# 面向对象编程
本节概述了 Scala 如何实现面向对象编程范式。到目前为止，你已经见过类（class），但 Scala 还具有以下特性：
- [抽象类](#abstract)
- [特质](#traits)
- [对象](#objects)
- [伴生对象](#compobj)
- [案例类](#caseclass)

## 抽象类<a name="abstract"></a>
抽象类与其他编程语言中的实现类似。它们可以定义许多需要子类实现的未实现值。每个对象只能直接继承一个父抽象类。

<span style="color:blue">**示例：抽象类**</span><br>

In [ ]:
abstract class MyAbstractClass {
  def myFunction(i: Int): Int
  val myValue: String
}
class ConcreteClass extends MyAbstractClass {
  def myFunction(i: Int): Int = i + 1
  val myValue = "Hello World!"
}
// Uncomment below to test!
// val abstractClass = new MyAbstractClass() // Illegal! Cannot instantiate an abstract class
val concreteClass = new ConcreteClass()      // Legal!


## 特质<a name="traits"></a>
特质与抽象类非常相似，它们都可以定义未实现的值。然而，它们有两个不同之处：
- 一个类可以继承多个特质
- 特质不能有构造参数

<span style="color:blue">**示例：特质与多继承**</span><br>
特质是 Scala 实现多重继承的方式，如下例所示。`MyClass` 同时继承了 `HasFunction` 和 `HasValue` 两个特质：

In [ ]:
trait HasFunction {
  def myFunction(i: Int): Int
}
trait HasValue {
  val myValue: String
  val myOtherValue = 100
}
class MyClass extends HasFunction with HasValue {
  override def myFunction(i: Int): Int = i + 1
  val myValue = "Hello World!"
}
// Uncomment below to test!
// val myTraitFunction = new HasFunction() // Illegal! Cannot instantiate a trait
// val myTraitValue = new HasValue()       // Illegal! Cannot instantiate a trait
val myClass = new MyClass()                // Legal!

要继承多个特质，像这样链式声明：

```scala
class MyClass extends HasTrait1 with HasTrait2 with HasTrait3 ...
```
一般来说，优先使用特质而非抽象类，除非你确定要强制使用抽象类的单继承限制。

## 对象<a name="objects"></a>
Scala 有一种针对单例类的语言特性，称为对象（object）。你不需要实例化一个对象（**不需要调用 `new`**）；你可以直接引用它。这使得它们类似于 Java 的静态类。

<span style="color:blue">**示例：对象**</span><br>

In [ ]:
object MyObject {
  def hi: String = "Hello World!"
  def apply(msg: String) = msg
}
println(MyObject.hi)
println(MyObject("This message is important!")) // equivalent to MyObject.apply(msg)

## 伴生对象<a name="compobj"></a>

当一个类和同一个文件中的对象共享相同名称时，该对象被称为**伴生对象**（companion object）。当你在类/对象名称前使用 `new` 时，它会实例化该类。如果不使用 `new`，则会引用该对象：

<span style="color:blue">**示例：伴生对象**</span><br>

In [ ]:
object Lion {
    def roar(): Unit = println("I'M AN OBJECT!")
}
class Lion {
    def roar(): Unit = println("I'M A CLASS!")
}
new Lion().roar()
Lion.roar()

伴生对象通常用于以下目的：
  1. 包含与类相关的常量
  2. 在类构造函数之前/之后执行代码
  3. 为类创建多个构造函数

在下面的示例中，我们将实例化多个 Animal 实例。我们希望每个动物都有一个名字，并知道它在所有实例化中的顺序。最后，如果没有给定名称，它应该获得一个默认名称。

In [ ]:
object Animal {
    val defaultName = "Bigfoot"
    private var numberOfAnimals = 0
    def apply(name: String): Animal = {
        numberOfAnimals += 1
        new Animal(name, numberOfAnimals)
    }
    def apply(): Animal = apply(defaultName)
}
class Animal(name: String, order: Int) {
  def info: String = s"Hi my name is $name, and I'm $order in line!"
}

val bunny = Animal.apply("Hopper") // Calls the Animal factory method
println(bunny.info)
val cat = Animal("Whiskers")       // Calls the Animal factory method
println(cat.info)
val yeti = Animal()                // Calls the Animal factory method
println(yeti.info)


*这里发生了什么？*
1. 我们的 **Animal 伴生对象** 定义了与 ```class Animal``` 相关的一个常量：
```scala
val defaultName = "Bigfoot"
```
1. 它还定义了一个私有的可变整数来跟踪 Animal 实例的顺序：
```scala 
private var numberOfAnimals = 0
```
1. 它定义了两个 **apply** 方法，这些方法被称为**工厂方法**，因为它们返回 **class Animal** 的实例。
    1. 第一个方法只使用一个参数 ```name``` 创建 Animal 实例，并使用 ```numberOfAnimals``` 来调用 Animal 类的构造函数。
```scala
def apply(name: String): Animal = {
            numberOfAnimals += 1
            new Animal(name, numberOfAnimals)
}
```
    1. 第二个工厂方法不需要参数，而是使用默认名称来调用另一个 apply 方法。
```scala
def apply(): Animal = apply(defaultName)
```
1. 这些工厂方法可以像这样直接调用：
```scala
val bunny = Animal.apply("Hopper")
```
这消除了使用 new 关键字的必要，但真正的魔法在于，编译器在看到任何实例或对象上带有括号时，都会自动假定调用了 apply 方法：
```scala
val cat = Animal("Whiskers")
```
1. 工厂方法通常通过伴生对象提供，允许以其他方式表达实例创建，为构造参数提供额外检查、类型转换，并消除使用关键字 ```new``` 的需要。请注意，你必须调用伴生对象的 `apply` 方法，`numberOfAnimals` 才会增加。

**Chisel 使用了许多伴生对象，比如 Module。** 当你编写以下代码时：
```scala
val myModule = Module(new MyModule)
```
你正在调用 **Module 伴生对象**，这样 Chisel 可以在实例化 ```MyModule``` 前后运行后台代码。

## 案例类<a name="caseclass"/>
案例类是一种特殊的 Scala 类，提供了一些很酷的额外特性。它们在 Scala 编程中非常常见，因此本节概述了它们的一些有用特性：
- 允许**外部访问**类的**参数**
- 实例化类时**无需**使用 **`new`**
- 自动创建一个 **unapply 方法**，提供对所有类参数的访问
- 不能作为父类被继承

在下面的示例中，我们声明了三个不同的类：`Nail`、`Screw` 和 `Staple`。

In [ ]:
class Nail(length: Int) // Regular class
val nail = new Nail(10) // Requires the `new` keyword
// println(nail.length) // Illegal! Class constructor parameters are not by default externally visible

class Screw(val threadSpace: Int) // By using the `val` keyword, threadSpace is now externally visible
val screw = new Screw(2)          // Requires the `new` keyword
println(screw.threadSpace)

case class Staple(isClosed: Boolean) // Case class constructor parameters are, by default, externally visible
val staple = Staple(false)           // No `new` keyword required
println(staple.isClosed)

`Nail` 是一个常规类，其参数对外部不可见，因为我们在参数列表中没有使用 `val` 关键字。声明 `Nail` 的实例时还需要使用 `new` 关键字。

`Screw` 的声明类似于 `Nail`，但在参数列表中包含了 `val`。这使其参数 `threadSpace` 对外部可见。

通过使用案例类，`Staple` 的所有参数都自动对外部可见（无需 `val` 关键字）。

此外，声明案例类时 `Staple` 不需要使用 `new`。这是因为 Scala 编译器会自动为代码中的每个案例类创建一个伴生对象，其中包含该案例类的 apply 方法。

案例类是包含大量参数的生成器的良好容器。
构造函数为你提供了一个定义派生参数和验证输入的好地方。

In [ ]:
case class SomeGeneratorParameters(
    someWidth: Int,
    someOtherWidth: Int = 10,
    pipelineMe: Boolean = false
) {
    require(someWidth >= 0)
    require(someOtherWidth >= 0)
    val totalWidth = someWidth + someOtherWidth
}

---
# 在 Chisel 中使用继承<a name="super"></a>
你已经见过 `Module` 和 `Bundle`，但理解其背后的本质很重要。
你创建的每个 Chisel 模块都是一个继承自基类型 `Module` 的类。
你创建的每个 Chisel IO 都是一个继承自基类型 `Bundle` 的类（或某些特殊情况下，`Bundle` 的父类型 [`Record`](https://github.com/freechipsproject/chisel3/blob/v3.0.0/chiselFrontend/src/main/scala/chisel3/core/Aggregate.scala#L415)）。
像 `UInt` 或 `Bundle` 这样的 Chisel 硬件类型都以 `Data` 作为父类型。
我们将探讨如何使用面向对象编程创建层次化的硬件模块，并探讨对象的复用。你将在下一个关于类型泛型生成器的模块中了解更多关于类型和 `Data` 的知识。

## Module<a name="module"></a>
无论何时你想在 Chisel 中创建一个硬件对象，它都需要以 `Module` 作为父类。
继承可能不总是复用的正确工具（[组合优于继承](https://en.wikipedia.org/wiki/Composition_over_inheritance)是一个常见原则），但继承仍然是一个强大的工具。
下面是一个创建 `Module` 并将它们以层次化方式连接在一起的示例。

<span style="color:blue">**示例：格雷编码器和解码器**</span><br>
我们将创建一个硬件格雷编码器/解码器。编码或解码操作的选择是硬件可编程的。

In [ ]:
import scala.math.pow

// create a module
class GrayCoder(bitwidth: Int) extends Module {
  val io = IO(new Bundle{
    val in = Input(UInt(bitwidth.W))
    val out = Output(UInt(bitwidth.W))
    val encode = Input(Bool()) // decode on false
  })
  
  when (io.encode) { //encode
    io.out := io.in ^ (io.in >> 1.U)
  } .otherwise { // decode, much more complicated
    io.out := Seq.fill(log2Ceil(bitwidth))(Wire(UInt(bitwidth.W))).zipWithIndex.fold((io.in, 0)){
      case ((w1: UInt, i1: Int), (w2: UInt, i2: Int)) => {
        w2 := w1 ^ (w1 >> pow(2, log2Ceil(bitwidth)-i2-1).toInt.U)
        (w2, i1)
      }
    }._1
  }
}


来试试吧！

In [ ]:
// test our gray coder
val bitwidth = 4
test(new GrayCoder(bitwidth)) { c =>
    def toBinary(i: Int, digits: Int = 8) = {
        String.format("%" + digits + "s", i.toBinaryString).replace(' ', '0')
    }
    println("Encoding:")
    for (i <- 0 until pow(2, bitwidth).toInt) {
        c.io.in.poke(i.U)
        c.io.encode.poke(true.B)
        c.clock.step(1)
        println(s"In = ${toBinary(i, bitwidth)}, Out = ${toBinary(c.io.out.peek().litValue.toInt, bitwidth)}")
    }

    println("Decoding:")
    for (i <- 0 until pow(2, bitwidth).toInt) {
        c.io.in.poke(i.U)
        c.io.encode.poke(false.B)
        c.clock.step(1)
        println(s"In = ${toBinary(i, bitwidth)}, Out = ${toBinary(c.io.out.peek().litValue.toInt, bitwidth)}")
    }

}

格雷码常用于异步接口中。通常使用格雷计数器而非功能完备的编码器/解码器，但我们使用上面的模块来简化问题。下面是一个 AsyncFIFO 示例，使用上述格雷编码器构建。控制逻辑和测试器留作后续练习。现在，请观察格雷编码器是如何多次实例化和连接的。

In [ ]:
class AsyncFIFO(depth: Int = 16) extends Module {
  val io = IO(new Bundle{
    // write inputs
    val write_clock = Input(Clock())
    val write_enable = Input(Bool())
    val write_data = Input(UInt(32.W))
    
    // read inputs/outputs
    val read_clock = Input(Clock())
    val read_enable = Input(Bool())
    val read_data = Output(UInt(32.W))
    
    // FIFO status
    val full = Output(Bool())
    val empty = Output(Bool())
  })
  
  // add extra bit to counter to check for fully/empty status
  assert(isPow2(depth), "AsyncFIFO needs a power-of-two depth!")
  val write_counter = withClock(io.write_clock) { Counter(io.write_enable && !io.full, depth*2)._1 }
  val read_counter = withClock(io.read_clock) { Counter(io.read_enable && !io.empty, depth*2)._1 }
  
  // encode
  val encoder = new GrayCoder(write_counter.getWidth)
  encoder.io.in := write_counter
  encoder.io.encode := true.B
  
  // synchronize
  val sync = withClock(io.read_clock) { ShiftRegister(encoder.io.out, 2) }
  
  // decode
  val decoder = new GrayCoder(read_counter.getWidth)
  decoder.io.in := sync
  decoder.io.encode := false.B
  
  // status logic goes here
  
}

---
# 你已完成！

[返回顶部](#top)